In [1]:
import os
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

/Users/apple/vscode/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. FUNCTION: Clean 10-K HTML
def parse_10k_html(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
        
    #we grab all text but remove script/style tags
    for script in soup(["script", "style"]):
        script.extract()
        
    # Get text and break into lines
    lines = (line.strip() for line in soup.get_text().splitlines())
    # Break multi-headlines into a line each
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    # Drop blank lines
    text = '\n'.join(chunk for chunk in chunks if chunk)
    
    # Split into sentences (simple rule-based split for speed)
    sentences = text.split('. ')
    return [s.replace('\n', ' ').strip() for s in sentences if len(s) > 20]


In [3]:
# 3. INDEXING: Process your 200 files
all_sentences = []
doc_map = [] # To track which file a sentence came from
folder_path = '/Users/apple/Downloads/OneDrive_1_02-02-2026/2010'

In [ ]:



print(f"\n📂 Processing year {year}...")
for filename in os.listdir(folder_path):
    if filename.endswith(".html") or filename.endswith(".htm"):
        path = os.path.join(folder_path, filename)
        try:
            # Extract sentences from this doc
            doc_sentences = parse_10k_html(path)
            
            # Add to our master list
            all_sentences.extend(doc_sentences)
            doc_map.extend([filename] * len(doc_sentences))
        except Exception as e:
            print(f"Skipping {filename}: {e}")

    print(f"Total sentences extracted: {len(all_sentences)}")

    # 4. EMBEDDING: Convert text to numbers
    # This might take a few minutes for 200 docs (approx 100k+ sentences)
    print("Embedding sentences (this occurs locally)...")
    # 4. EMBEDDING & INDEXING
    embeddings = model.encode(all_sentences, normalize_embeddings=True)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    # 5. QUERY PREPARATION
    query_text = "high competition"
    query_vector = model.encode([query_text], normalize_embeddings=True)

    # ==========================================================
    # NEW SECTION: AUTOMATIC ELBOW CALCULATION
    # ==========================================================
    print("Calculating dynamic threshold using Elbow Method...")

    # Step A: Probe the data (Get top 100 matches to analyze the curve)
    k_probe = min(100, len(all_sentences)) 
    dists, idxs = index.search(query_vector, k_probe)

    # Step B: Convert to Similarity Scores
    # Formula: Similarity = 1 - (Distance^2 / 2)
    sim_scores = 1 - (dists[0] / 2)

    # Step C: Find the Elbow Point (Geometric Method)
    # We find the point on the curve with the max distance to the 
    # line connecting the first point (highest score) and the last point (k_probe).

    # Coordinates of the curve
    x_curve = np.arange(len(sim_scores))
    y_curve = sim_scores

    # Line endpoints
    p1 = np.array([0, y_curve[0]])
    p2 = np.array([len(y_curve)-1, y_curve[-1]])

    # Calculate distance from each point on curve to the line defined by p1 and p2
    # Vector math: |cross_product| / magnitude_of_line
    p2_p1 = p2 - p1
    distances_to_line = []
    for i in range(len(y_curve)):
        p0 = np.array([i, y_curve[i]])
        dist = np.abs(np.cross(p2_p1, p1 - p0)) / np.linalg.norm(p2_p1)
        distances_to_line.append(dist)

    # The index with the max distance is the "Elbow"
    elbow_idx = np.argmax(distances_to_line)
    calculated_threshold = sim_scores[elbow_idx]

    # Safety Clamp: Don't let it go too low (e.g., below 0.50) or too high
    calculated_threshold = max(0.3, min(calculated_threshold, 0.85))

    print(f"--> Elbow found at Rank {elbow_idx}")
    print(f"--> Calculated Similarity Threshold: {calculated_threshold:.4f}")

    # ==========================================================
    # NEW SECTION: RADIUS SEARCH
    # ==========================================================

    # Step D: Convert Similarity Threshold back to L2 Radius
    # Formula: Radius = 2 * (1 - Similarity)
    search_radius = 2 * (1 - calculated_threshold)
    print(f"--> Search Radius: {search_radius:.4f}\n")

    # Step E: Perform the Definite Search
    limits, distances, indices = index.range_search(query_vector, search_radius)

    # ==========================================================
    # OUTPUT
    # ==========================================================
    print(f"--- MATCHES FOR '{query_text}' ---")
    match_count = limits[1] - limits[0]
    print(f"Found {match_count} definite matches:\n")

    # Collect results
    results = []
    for i in range(limits[0], limits[1]):
        idx = indices[i]
        dist = distances[i]
        score = 1 - (dist / 2)
        results.append((score, all_sentences[idx]))

    # Sort by score descending
    results.sort(key=lambda x: x[0], reverse=True)

    for score, text in results:
        print(f"[{score:.1%}] {text}")


📂 Processing year ...
Total sentences extracted: 1683
Embedding sentences (this occurs locally)...
Calculating dynamic threshold using Elbow Method...
--> Elbow found at Rank 18
--> Calculated Similarity Threshold: 0.3000
--> Search Radius: 1.4000

--- MATCHES FOR 'high competition' ---
Found 4 definite matches:

[36.9%] Our ability to compete depends on our ability to develop, introduce and sell new products or enhanced versions of existing products on a timely basis and at competitive prices, while reducing our costs. Competition in the Microprocessor Market Intel Corporation has dominated the market for microprocessors for many years
[35.1%] Our ability to develop and qualify new products and related technologies to meet evolving industry requirements, at prices acceptable to our customers and on a timely basis are significant factors in determining our competitiveness in our target markets
[33.2%] We believe that the main factors that determine our product competitiveness are time

KeyboardInterrupt: 